# PhoBERT v6: adapt v3 to guideline v3.2

Setup: **GPU T4**, **Internet On**, secret `HF_TOKEN`.
**Input:** one private dataset with two files: `v6_train.csv` and `v6_val.csv` (1,739 comments from 2026
labeled under guideline v3.2; columns `text,label`).

Instead of retraining on the 52k older labels, this continues training the published v3 model for 3 epochs
on the new labels only (learning rate 1e-5). The older labels call many harsh-but-clean comments toxic and
use the previous hate/offensive boundary, which is what v2-v5 kept reproducing.

The model is pushed to the **`v6` branch** (`main` is untouched). Takes about 5 minutes.


In [ ]:
!git clone -q https://github.com/Harukaru1/vietnamese-toxic-comment-moderation.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pyvi emoji
import glob, os, shutil, json
os.makedirs("data/adapt_v6", exist_ok=True)
for split in ["train", "val"]:
    matches = glob.glob(f"/kaggle/input/**/v6_{split}.csv", recursive=True)
    assert len(matches) == 1, f"Expected exactly one v6_{split}.csv, found: {matches}"
    shutil.copy(matches[0], f"data/adapt_v6/{split}.csv")
shutil.copy("data/adapt_v6/val.csv", "data/adapt_v6/test.csv")  # the real test half never leaves the author's machine
os.makedirs("reports", exist_ok=True)


In [ ]:
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import whoami
print("Logged in as:", whoami()["name"])

In [ ]:
from huggingface_hub import whoami
repo_id = f"{whoami()['name']}/phobert-vi-toxic-comment"
!python scripts/train_phobert.py --model {repo_id} --data-dir data/adapt_v6 --output-dir models/v6 --epochs 3 --lr 1e-5 --batch-size 16 --max-len 128 --tag _v6 --no-push


In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_branch(repo_id, branch="v6", exist_ok=True)
api.upload_folder(folder_path="models/v6/best", repo_id=repo_id, revision="v6", commit_message="v6: v3 adapted to guideline v3.2 (1,739 comments from 2026)")
print(f"https://huggingface.co/{repo_id}/tree/v6")


Copy the output of the next cell and paste it into the chat.

In [ ]:
print(json.dumps(json.load(open("reports/phobert_results_v6.json")), ensure_ascii=False))
